# RNN/LSTM: 상태를 계산하고 반환값을 읽기

방금 확인한 RNN의 재귀 상태, 시퀀스 반환 shape, LSTM gate를 작은 수치와 실제 PyTorch 텐서로 다시 다룹니다. 각 구현 셀의 TODO만 채운 뒤 fixture와 `check_e01()`, `check_e02()`, `check_e03()`을 차례로 실행하세요. 검사가 통과한 뒤에는 각 결과 해석을 자신의 말로 작성합니다.


In [1]:
import math

import torch


## 시간 순서대로 RNN hidden state 계산하기

### 왜 필요한가

RNN은 같은 가중치를 매 시점에 쓰지만, 바로 앞 hidden state를 다음 계산에 함께 넣습니다. 따라서 입력이 같은 집합이어도 순서가 바뀌면 hidden state의 흐름이 달라질 수 있습니다.

### 작은 수치 추적

이번 함수는 scalar 입력만 사용합니다. 각 시점에서 `input_weight * x + recurrent_weight * h + bias`를 계산하고 ReLU를 적용합니다. 예를 들어 `h=0`, `x=-2`, 모든 가중치가 `1`, bias가 `0`이면 다음 hidden state는 `0`입니다.

### 주어진 규칙

- `rnn_relu_trace`는 각 입력을 시간 순서대로 처리하며 `ReLU(input_weight * x_t + recurrent_weight * h_(t-1) + bias)`로 다음 hidden state를 계산해야 합니다.
- `rnn_relu_trace`는 입력마다 계산된 hidden state를 같은 시간 순서의 tuple로 반환해야 하며, 빈 입력은 빈 tuple을 반환해야 합니다.
- 결과 해석에서는 ReLU가 첫 시점의 음수 pre-activation을 0으로 만든 뒤 다음 시점 계산에 어떤 hidden state가 전달되는지 설명해야 합니다.

<details><summary>힌트 1</summary>

loop의 시작에서 `h`는 바로 직전 시점의 hidden state입니다. 먼저 pre-activation 하나를 만든 뒤 ReLU를 적용하세요.

</details>

<details><summary>힌트 2</summary>

scalar ReLU는 `max(0.0, value)`로 쓸 수 있습니다. 갱신한 `h`를 list에 추가한 뒤 마지막에 tuple로 바꾸세요.

</details>

### 해보기

아래 TODO에서 한 시점의 pre-activation을 완성하세요. loop와 반환 형식은 바꾸지 마세요.

### 확인하기

fixture와 `check_e01()`로 음수가 0으로 잘리는 세 시점 trace와 빈 입력을 확인하세요.

### 결과 해석

검사 뒤 빈칸에 2~4문장으로, 첫 음수 pre-activation 뒤 다음 시점에 전달되는 hidden state를 설명하세요.


In [ ]:
def rnn_relu_trace(
    inputs: list[float],
    *,
    input_weight: float,
    recurrent_weight: float,
    bias: float,
    h0: float,
) -> tuple[float, ...]:
    """Return one scalar ReLU-RNN hidden state for every input step."""
    h = float(h0)
    states: list[float] = []
    for x in inputs:
        # TODO: 현재 입력과 직전 hidden state를 함께 써서 pre-activation을 계산하세요.
        pre_activation = x + (0 if not states else states[-1])
        h = max(0.0, pre_activation)
        states.append(float(h))
    return tuple(states)


In [6]:
trace = rnn_relu_trace([-2.0, 2.0, 0.0], input_weight=1.0, recurrent_weight=1.0, bias=0.0, h0=0.0)
print(trace)


(0.0, 0.0, 0.0)


In [4]:
def check_e01() -> None:
    actual = torch.tensor(rnn_relu_trace([-2.0, 2.0, 0.0], input_weight=1.0, recurrent_weight=1.0, bias=0.0, h0=0.0))
    torch.testing.assert_close(actual, torch.tensor([0.0, 2.0, 2.0]))
    torch.testing.assert_close(torch.tensor(rnn_relu_trace([], input_weight=1.0, recurrent_weight=1.0, bias=0.0, h0=3.0)), torch.tensor([]))


check_e01()


AssertionError: Tensor-likes are not close!

Mismatched elements: 2 / 3 (66.7%)
Greatest absolute difference: 2.0 at index (1,) (up to 1e-05 allowed)
Greatest relative difference: 1.0 at index (1,) (up to 1.3e-06 allowed)

### 결과 해석

<!-- TODO: 첫 음수 pre-activation 뒤 다음 시점에 전달되는 hidden state를 2~4문장으로 설명하세요. -->
<여기에 2~4문장으로 작성하세요>


## RNN과 LSTM이 반환하는 shape 읽기

### 왜 필요한가

시퀀스 출력에는 모든 time step 축이 남지만 final state에는 그 축이 없습니다. token별 예측과 문장 하나의 분류를 구분하려면 이 차이를 코드에서 바로 읽을 수 있어야 합니다.

### 작은 예

`batch_first=True`, one-layer, one-direction이고 batch가 `3`, step이 `5`, hidden size가 `2`라고 합시다. RNN `output`은 `(3, 5, 2)`이고 `h_n`은 `(1, 3, 2)`입니다. LSTM은 여기에 같은 shape의 `c_n`이 하나 더 있습니다.

### 주어진 규칙

- one-layer, one-direction, `batch_first=True` RNN의 `sequence_return_shapes` 결과는 `((batch_size, steps, hidden_size), (1, batch_size, hidden_size))`여야 합니다.
- one-layer, one-direction, `batch_first=True` LSTM의 `sequence_return_shapes` 결과는 `((batch_size, steps, hidden_size), (1, batch_size, hidden_size), (1, batch_size, hidden_size))`여야 합니다.
- `kind`가 `RNN` 또는 `LSTM`이 아니거나 세 size 중 하나라도 1보다 작으면 `ValueError`를 발생시켜야 합니다.
- 결과 해석에서는 `output`에만 sequence axis가 남는 이유, token별 예측에 `output`이 필요한 이유, 그리고 양수 size 경계가 필요한 이유를 설명해야 합니다.

<details><summary>힌트 1</summary>

`batch_first=True`일 때 sequence output의 축 순서는 batch, step, hidden입니다. final state의 첫 축은 layer와 direction을 합친 축입니다.

</details>

<details><summary>힌트 2</summary>

이번에는 layer도 direction도 하나이므로 final state의 첫 원소는 `1`입니다. LSTM의 `h_n`과 `c_n`은 이 설정에서 같은 shape를 가집니다.

</details>

### 해보기

RNN과 LSTM 각각의 반환 tuple을 TODO에 적으세요. 제공한 입력 검사는 그대로 두세요.

### 확인하기

fixture는 실제 PyTorch RNN/LSTM의 shape를 출력합니다. 그 다음 `check_e02()`로 계산한 tuple, 최소 크기, 잘못된 kind와 0-size를 확인하세요.

### 결과 해석

검사 뒤 빈칸에 2~4문장으로, `output`에만 sequence axis가 남는 이유, token별 예측에서 `output`을 쓰는 이유, 양수 size 경계가 필요한 이유를 적으세요.


In [ ]:
def sequence_return_shapes(kind: str, batch_size: int, steps: int, hidden_size: int) -> tuple[tuple[int, ...], ...]:
    """Describe one-layer, one-direction batch-first RNN/LSTM returns."""
    if kind not in {"RNN", "LSTM"}:
        raise ValueError("kind must be RNN or LSTM")
    if min(batch_size, steps, hidden_size) < 1:
        raise ValueError("all sizes must be positive")

    # TODO: RNN의 output과 h_n shape를 순서대로 tuple에 담으세요.
    rnn_shapes = NotImplemented
    # TODO: LSTM의 output, h_n, c_n shape를 순서대로 tuple에 담으세요.
    lstm_shapes = NotImplemented
    return rnn_shapes if kind == "RNN" else lstm_shapes


In [ ]:
torch.manual_seed(0)
x = torch.zeros((3, 5, 2))
rnn_output, rnn_h_n = torch.nn.RNN(input_size=2, hidden_size=2, batch_first=True)(x)
lstm_output, (lstm_h_n, lstm_c_n) = torch.nn.LSTM(input_size=2, hidden_size=2, batch_first=True)(x)
print(tuple(rnn_output.shape), tuple(rnn_h_n.shape))
print(tuple(lstm_output.shape), tuple(lstm_h_n.shape), tuple(lstm_c_n.shape))


In [ ]:
def check_e02() -> None:
    torch.testing.assert_close(torch.tensor(sequence_return_shapes("RNN", 3, 5, 2)), torch.tensor([[3, 5, 2], [1, 3, 2]]))
    torch.testing.assert_close(torch.tensor(sequence_return_shapes("LSTM", 3, 5, 2)), torch.tensor([[3, 5, 2], [1, 3, 2], [1, 3, 2]]))
    torch.testing.assert_close(torch.tensor(sequence_return_shapes("RNN", 1, 1, 1)), torch.tensor([[1, 1, 1], [1, 1, 1]]))
    try:
        sequence_return_shapes("GRU", 3, 5, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("unknown kind must raise ValueError")
    try:
        sequence_return_shapes("RNN", 0, 1, 1)
    except ValueError:
        pass
    else:
        raise AssertionError("nonpositive size must raise ValueError")


check_e02()


### 결과 해석

<!-- TODO: output에만 sequence axis가 남는 이유, token별 예측에서 output을 쓰는 이유, 양수 size 경계가 필요한 이유를 2~4문장으로 설명하세요. -->
<여기에 2~4문장으로 작성하세요>


## LSTM cell state의 한 단계 갱신하기

### 왜 필요한가

LSTM은 cell state에 이전 기억을 그대로 더하는 장치가 아니라, forget gate와 input gate가 정한 비율로 이전 기억과 새 candidate를 섞습니다. 따라서 gate 값이 기억 경로를 직접 바꿉니다.

### 작은 수치 추적

`c_prev=1`, `forget=0.2`, `input_gate=0.5`, `candidate=2`, `output_gate=1`이면 cell state는 `0.2 * 1 + 0.5 * 2`가 됩니다. hidden state는 그 cell state에 tanh와 output gate를 적용한 값입니다.

### 주어진 규칙

- `lstm_cell_step`는 `(forget * c_prev + input_gate * candidate, output_gate * tanh(cell_state))`를 `(cell_state, hidden_state)` 순서로 반환해야 합니다.
- `forget`, `input_gate`, `output_gate` 중 하나라도 0보다 작거나 1보다 크면 `ValueError`를 발생시켜야 합니다.
- 결과 해석에서는 forget gate가 0이면 이전 `c_prev`의 직접 기여가 왜 사라지는지와 이것이 LSTM이 무조건 기억하는 장치가 아님을 어떻게 보여 주는지 설명해야 합니다.

<details><summary>힌트 1</summary>

먼저 cell state의 두 항을 각각 계산해 더하세요. `candidate`는 이 함수에 이미 들어온 값이므로 여기서 다시 tanh를 적용하지 않습니다.

</details>

<details><summary>힌트 2</summary>

hidden state에는 완성한 cell state를 넣어 `math.tanh(cell_state)`를 계산한 뒤 output gate를 곱하세요.

</details>

### 해보기

TODO에서 cell state와 hidden state를 함께 완성하세요. 제공된 gate 범위 검사는 바꾸지 마세요.

### 확인하기

fixture와 `check_e03()`로 위 작은 수치와 forget gate가 0인 경우, 범위를 벗어난 gate를 확인하세요.

### 결과 해석

검사 뒤 빈칸에 2~4문장으로 forget gate가 0일 때 사라지는 연결과 LSTM의 한계를 설명하세요.


In [ ]:
def lstm_cell_step(
    c_prev: float,
    forget: float,
    input_gate: float,
    candidate: float,
    output_gate: float,
) -> tuple[float, float]:
    """Return one scalar LSTM cell state and exposed hidden state."""
    if not all(0.0 <= gate <= 1.0 for gate in (forget, input_gate, output_gate)):
        raise ValueError("gate values must lie in [0, 1]")

    # TODO: forget와 input gate를 적용한 cell state, 그리고 output gate를 적용한 hidden state를 계산하세요.
    cell_state, hidden_state = NotImplemented, NotImplemented
    return float(cell_state), float(hidden_state)


In [ ]:
cell_state, hidden_state = lstm_cell_step(c_prev=1.0, forget=0.2, input_gate=0.5, candidate=2.0, output_gate=1.0)
print(cell_state, hidden_state)


In [ ]:
def check_e03() -> None:
    torch.testing.assert_close(torch.tensor(lstm_cell_step(c_prev=1.0, forget=0.2, input_gate=0.5, candidate=2.0, output_gate=1.0)), torch.tensor([1.2, math.tanh(1.2)]))
    torch.testing.assert_close(torch.tensor(lstm_cell_step(c_prev=9.0, forget=0.0, input_gate=0.5, candidate=2.0, output_gate=1.0)), torch.tensor([1.0, math.tanh(1.0)]))
    try:
        lstm_cell_step(c_prev=1.0, forget=1.2, input_gate=0.5, candidate=2.0, output_gate=1.0)
    except ValueError:
        pass
    else:
        raise AssertionError("out-of-range gate must raise ValueError")


check_e03()


### 결과 해석

<!-- TODO: forget gate가 0일 때 사라지는 연결과 LSTM이 무조건 기억하지 않는 이유를 2~4문장으로 설명하세요. -->
<여기에 2~4문장으로 작성하세요>
